# Practical Machine Learning — Regression Day
**CDAC Kharghar · PML · 26 October 2026**
### Student Exercise Notebook
Work through exercises **A1 → A11** in order. Every `TODO` marks code *you* must write. Where you see `...`, replace it with working code.
Run cells top to bottom (`Shift + Enter`). If something breaks, re-run from the setup cell.

**Datasets** (synthetic, generated for this course): `regression_manual_example.csv`, `house_price_regression.csv`, `polynomial_regression.csv`, `delivery_time_challenge.csv` in the `04_Datasets` folder.

**Ground rules used all day**
- One fixed split: `train_test_split(..., test_size=0.2, random_state=42)` so every model is compared on the *same* held-out rows.
- Preprocessing (scaling, polynomial features) lives *inside* a `Pipeline`, so it is fitted on training data only.
- The test set is used **once per final model**, never to tune hyperparameters; tuning uses cross-validation on the training set.

In [ ]:
# ---- Setup: run this first ----
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True, "grid.alpha": 0.3})

# Locate the datasets folder whether the notebook is opened from 03_Jupyter_Notebooks or the package root
candidates = [Path("../04_Datasets"), Path("04_Datasets"), Path("."), Path("data")]
DATA = next(p for p in candidates if (p / "house_price_regression.csv").exists())
print("Datasets folder:", DATA.resolve())

---
## A1 · Dataset understanding
**Objective:** identify the features (X) and the target (y) before building any model.

**Concept reminder:** in supervised regression, the *target* is the continuous quantity we want to predict; *features* are the inputs we predict it from. A feature must be available *at prediction time* — otherwise it leaks the answer.

**Dataset:** `house_price_regression.csv` — 300 synthetic houses. Price is in **₹ lakh** (1 lakh = ₹100,000).

**Instructions**
1. Load the CSV into `house`.
2. Show the first 5 rows, the shape, and `describe()`.
3. Create `FEATURES` (list of the six input column names) and `TARGET`.

**Expected output:** a 300 × 7 DataFrame; summary statistics table.

In [ ]:
house = ...          # TODO: read DATA / "house_price_regression.csv"
# TODO: display house.head(), house.shape and house.describe()

In [ ]:
TARGET = ...          # TODO: name of the target column (string)
FEATURES = ...        # TODO: list of the six feature column names
print("Target  :", TARGET)
print("Features:", FEATURES)

**Questions**
1. Which features would you expect to *increase* price, and which to *decrease* it?
2. `parking` takes only the values 0 and 1. What kind of variable is it, and can linear regression still use it?
3. Suppose the file also had a column `price_per_sqft`. Why must it never be used as a feature?

**Reflection:** what one extra real-world feature would you add to this dataset, and why?
*Write your answers here.*

---
## A2 · Simple Linear Regression — a baseline
**Objective:** fit $\hat y = b_0 + b_1x$ with one feature (`area_sqft`) and look at it.

**Concept reminder:** the model draws the straight line with the smallest sum of squared residuals on the *training* data.

**Instructions**
1. Split **all features** and the target once: `test_size=0.2, random_state=42`. (We reuse this split in A7 and A10 so comparisons are fair.)
2. Fit `LinearRegression()` on the `area_sqft` column only (keep it 2-D: `X_train[["area_sqft"]]`).
3. Plot the test points and the fitted line.

**Expected output:** shapes (240, 6) and (60, 6); a scatter plot with a rising line.

In [ ]:
X_train, X_test, y_train, y_test = ...   # TODO: train_test_split(house[FEATURES], house[TARGET], test_size=0.2, random_state=SEED)
print(X_train.shape, X_test.shape)

In [ ]:
slr = ...                     # TODO: create a LinearRegression model
# TODO: fit slr using X_train[["area_sqft"]] and y_train

grid = pd.DataFrame({"area_sqft": np.linspace(450, 2500, 50)})
plt.scatter(X_test["area_sqft"], y_test, s=18, label="test houses")
plt.plot(grid["area_sqft"], slr.predict(grid), color="C1", lw=2.5, label="fitted line")
plt.xlabel("area (sq ft)"); plt.ylabel("price (₹ lakh)"); plt.legend(); plt.title("Simple linear regression")
plt.show()

**Questions**
1. Why does scikit-learn insist that `X` is 2-D even when there is a single feature?
2. Looking at the plot, name one reason the points do not lie exactly on the line.

**Reflection:** is area alone a good predictor? What would you check next?
*Write your answers here.*

---
## A3 · Interpreting slope and intercept
**Objective:** translate `coef_` and `intercept_` into plain language.

**Concept reminder:** $b_1$ = change in predicted $y$ for a one-unit increase in $x$. $b_0$ = predicted $y$ when $x=0$.

**Instructions**
1. Print `slr.coef_[0]` and `slr.intercept_`.
2. Compute *by hand* (using the printed numbers) the predicted price of a 1,200 sq ft house, then confirm with `slr.predict`.

In [ ]:
b1 = ...   # TODO: slope from slr
b0 = ...   # TODO: intercept from slr
print(f"slope b1     = {b1:.5f} ₹ lakh per sq ft")
print(f"intercept b0 = {b0:.3f} ₹ lakh")

In [ ]:
manual_1200 = ...   # TODO: b0 + b1 * 1200
model_1200 = ...    # TODO: slr.predict on a one-row DataFrame with area_sqft = 1200
print(f"manual: {manual_1200:.3f}   model: {model_1200:.3f}   equal: {np.isclose(manual_1200, model_1200)}")

**Questions**
1. Complete the sentence: "Each additional 100 sq ft is associated with a predicted price change of ___ ₹ lakh."
2. Interpret the intercept. Is it meaningful here?
3. Can we conclude that adding area to a house *causes* this price increase?
*Write your answers here.*

---
## A4 · Manual OLS — compute the line yourself
**Objective:** reproduce the least-squares slope and intercept using only NumPy and the formulas from class.

$$b_1=\frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2}\qquad b_0=\bar y-b_1\bar x$$

**Dataset:** `regression_manual_example.csv` — 5 deliveries: distance (km) and delivery time (min). *A teaching example: we fit and evaluate on the same five points, which shows the arithmetic but says nothing about performance on new data.*

**Expected output:** a table of deviations; slope 2.1 and intercept 8.4.

In [ ]:
manual = pd.read_csv(DATA / "regression_manual_example.csv")
x = manual["distance_km"].to_numpy(dtype=float)
y = manual["delivery_min"].to_numpy(dtype=float)
manual

In [ ]:
x_bar = ...                 # TODO: mean of x
y_bar = ...                 # TODO: mean of y
dx = ...                    # TODO: deviations x - x_bar
dy = ...                    # TODO: deviations y - y_bar
numerator = ...             # TODO: sum of dx * dy
denominator = ...           # TODO: sum of dx ** 2
b1_manual = ...             # TODO
b0_manual = ...             # TODO
table = pd.DataFrame({"x": x, "y": y, "x - x̄": dx, "y - ȳ": dy,
                      "(x-x̄)(y-ȳ)": dx * dy, "(x-x̄)²": dx ** 2})
display(table)
print(f"x̄ = {x_bar}, ȳ = {y_bar}, numerator = {numerator}, denominator = {denominator}")
print(f"b1 = {b1_manual:.4f}, b0 = {b0_manual:.4f}")

In [ ]:
y_hat = ...          # TODO: predictions from b0_manual and b1_manual
residuals = ...      # TODO: y - y_hat
print("predictions:", np.round(y_hat, 3))
print("residuals  :", np.round(residuals, 3), " | sum =", round(residuals.sum(), 10))
print("SSE        :", round(np.sum(residuals ** 2), 4))

# Verification against scikit-learn and against a least-squares matrix solver
check = LinearRegression().fit(x.reshape(-1, 1), y)
X_design = np.column_stack([np.ones_like(x), x])          # n x 2 design matrix [1, x]
beta_lstsq, *_ = np.linalg.lstsq(X_design, y, rcond=None)
print("sklearn  :", round(check.intercept_, 4), round(check.coef_[0], 4))
print("lstsq    :", np.round(beta_lstsq, 4))
assert np.isclose(check.coef_[0], b1_manual) and np.isclose(check.intercept_, b0_manual)

**Questions**
1. The residuals add up to (numerically) zero. Is that a coincidence? Use the derivative $\partial SSE/\partial b_0 = 0$ to explain.
2. Which single observation has the largest absolute residual?
3. If every delivery time were increased by 5 minutes, how would $b_1$ and $b_0$ change?

**Reflection:** which step of the calculation would be most error-prone by hand?
*Write your answers here.*

---
## A5 · Gradient Descent — complete the update steps
**Objective:** implement batch gradient descent for $\hat y = wx+b$ and watch the cost fall.

$$J(w,b)=\frac{1}{2n}\sum(y_i-\hat y_i)^2,\quad \frac{\partial J}{\partial w}=-\frac1n\sum(y_i-\hat y_i)x_i,\quad \frac{\partial J}{\partial b}=-\frac1n\sum(y_i-\hat y_i)$$
$$w \leftarrow w-\alpha\,\partial J/\partial w,\qquad b \leftarrow b-\alpha\,\partial J/\partial b$$

**Dataset:** the class example $x=[1,2,3]$, $y=[2,4,5]$, start $w=b=0$, $\alpha=0.1$.

**Expected output:** iteration 0 cost 7.5, iteration 1 cost ≈ 1.5098, iteration 2 cost ≈ 0.3217 — the same numbers we computed by hand.

In [ ]:
x_gd = np.array([1.0, 2.0, 3.0])
y_gd = np.array([2.0, 4.0, 5.0])

def compute_cost(w, b, x, y):
    ...   # TODO: return (1 / (2n)) * sum of squared residuals

def compute_gradients(w, b, x, y):
    ...   # TODO: return dJ_dw, dJ_db

def gradient_descent(x, y, w=0.0, b=0.0, alpha=0.1, iterations=100):
    history = []
    for it in range(iterations + 1):
        cost = compute_cost(w, b, x, y)
        dJ_dw, dJ_db = compute_gradients(w, b, x, y)
        history.append({"iteration": it, "w": w, "b": b, "cost": cost, "dJ_dw": dJ_dw, "dJ_db": dJ_db})
        w = ...   # TODO: update rule for w
        b = ...   # TODO: update rule for b
    return pd.DataFrame(history)

In [ ]:
hist = gradient_descent(x_gd, y_gd, alpha=0.1, iterations=200)
display(hist.head(4).round(4))
print("after 200 iterations:", hist.iloc[-1][["w", "b", "cost"]].round(4).to_dict())

ols = LinearRegression().fit(x_gd.reshape(-1, 1), y_gd)
print("least-squares solution: w =", round(ols.coef_[0], 4), " b =", round(ols.intercept_, 4))

plt.plot(hist["iteration"][:30], hist["cost"][:30], marker="o", ms=3)
plt.xlabel("iteration"); plt.ylabel("cost J"); plt.title("Gradient descent: cost per iteration"); plt.show()

In [ ]:
# Experiment: change the learning rate and compare
for alpha in [0.01, 0.1, 0.3, 0.37]:
    h = gradient_descent(x_gd, y_gd, alpha=alpha, iterations=50)
    print(f"alpha={alpha:<5} final cost after 50 iterations = {h['cost'].iloc[-1]:.6g}")

**Questions**
1. At iteration 0 both gradients are negative. Why does that make $w$ and $b$ *increase*?
2. Which learning rate diverges? Explain what happens to the updates.
3. scikit-learn's `LinearRegression` gave the answer instantly. Does it run this loop internally?

**Reflection:** when would you prefer an iterative method over the closed-form solution?
*Write your answers here.*

---
## A6 · Evaluation metrics
**Objective:** compute MAE, MSE, RMSE and R² by hand, confirm with scikit-learn, then evaluate the A2 model on **held-out** houses.

| metric | formula | units |
|---|---|---|
| MAE | $\frac1n\sum|y-\hat y|$ | same as y |
| MSE | $\frac1n\sum(y-\hat y)^2$ | y² |
| RMSE | $\sqrt{MSE}$ | same as y |
| R² | $1-SSE/SST$ | none |

In [ ]:
def regression_report(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    mae = ...    # TODO
    mse = ...    # TODO
    rmse = ...   # TODO
    sse = ...    # TODO: sum of squared residuals
    sst = ...    # TODO: sum of squared deviations of y_true from its mean
    r2 = ...     # TODO
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

ours = regression_report(y, y_hat)          # teaching example from A4 (training points!)
theirs = {"MAE": mean_absolute_error(y, y_hat), "MSE": mean_squared_error(y, y_hat),
          "RMSE": np.sqrt(mean_squared_error(y, y_hat)), "R2": r2_score(y, y_hat)}
pd.DataFrame({"manual": ours, "sklearn": theirs}).round(4)

In [ ]:
# Outlier experiment: identical errors except one large miss
actual  = np.array([10, 12, 15, 20, 25.0])
model_P = np.array([11, 11, 16, 19, 26.0])     # every error = 1
model_Q = np.array([11, 11, 16, 19, 35.0])     # four errors = 1, one error = 10
pd.DataFrame({"P": regression_report(actual, model_P), "Q": regression_report(actual, model_Q)}).round(3)

In [ ]:
slr_test_pred = ...   # TODO: predictions of slr (A2) on X_test[["area_sqft"]]
slr_scores = ...      # TODO: regression_report on y_test
pd.Series(slr_scores).round(3)

**Questions**
1. In the outlier experiment MAE grew 2.8×. By what factor did MSE grow? Why so much more?
2. The house model's RMSE is in what units? Express it as a sentence a manager would understand.
3. Could R² on the test set ever be negative? What would that mean?

**Reflection:** which metric would you report to a non-technical stakeholder, and which would you use to choose between models?
*Write your answers here.*

---
## A7 · Multiple Linear Regression
**Objective:** use all six features: $\hat y = b_0 + b_1x_1 + \dots + b_6x_6$, interpret coefficients and compare with A2 **on the same test set**.

**Concept reminder:** each $b_j$ is the change in predicted price per unit of $x_j$ **holding the other features fixed**. Coefficients are in different units, so their sizes are not directly comparable.

In [ ]:
mlr = ...          # TODO: LinearRegression fitted on X_train (all FEATURES), y_train
coef_table = pd.DataFrame({"feature": FEATURES, "coefficient": mlr.coef_,
                           "unit": ["₹ lakh per sq ft", "₹ lakh per bedroom", "₹ lakh per year",
                                    "₹ lakh per km", "₹ lakh per floor", "₹ lakh (has vs no parking)"]})
display(coef_table.round(4))
print("intercept:", round(mlr.intercept_, 3))

In [ ]:
def adjusted_r2(r2, n, p):
    # defined only when n - p - 1 > 0
    return ...   # TODO

mlr_scores = regression_report(y_test, mlr.predict(X_test))
mlr_scores["Adj R2"] = adjusted_r2(mlr_scores["R2"], n=len(y_test), p=len(FEATURES))
comparison = pd.DataFrame({"SLR (area only)": slr_scores, "MLR (6 features)": mlr_scores}).round(3)
comparison

In [ ]:
# Predict one new house — first by hand from coefficients, then with the model
new_house = pd.DataFrame([{"area_sqft": 1200, "bedrooms": 2, "property_age_yrs": 10,
                           "distance_city_km": 8, "floor_number": 5, "parking": 1}])
by_hand = ...      # TODO: intercept + sum(coef * value)
print(f"by hand: {by_hand:.3f}   model: {mlr.predict(new_house)[0]:.3f}")

display(house[FEATURES].corr().round(2))

**Questions**
1. Interpret the `distance_city_km` coefficient in one sentence, including "holding other features fixed".
2. Compare the area coefficient here with A2. Why *can* adding a correlated feature like `bedrooms` change a coefficient, even though here the change is small?
3. `area_sqft` and `bedrooms` have correlation ≈ 0.8. What problem can this cause for coefficient interpretation?
4. Does the large area coefficient mean area is the "most important" feature? Why/why not?
*Write your answers here.*

---
## A8 · Polynomial Regression
**Objective:** build polynomial features by hand, then with `PolynomialFeatures`, and compare degrees on curved data.

**Concept reminder:** $\hat y=b_0+b_1x+b_2x^2$ is curved in $x$ but **linear in the coefficients** — so ordinary least squares still fits it once we add the column $x^2$.

**Dataset:** `polynomial_regression.csv` — speed (km/h) vs fuel efficiency (km/l), 150 synthetic trips.

In [ ]:
# Step 1 — by hand on three values
x_small = np.array([[1.0], [2.0], [3.0]])
manual_features = ...    # TODO: columns [x, x^2] using np.column_stack
sk_features = PolynomialFeatures(degree=2, include_bias=False).fit_transform(x_small)
print(manual_features); print(sk_features)
print("identical:", np.allclose(manual_features, sk_features))

In [ ]:
poly_df = pd.read_csv(DATA / "polynomial_regression.csv")
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    poly_df[["speed_kmph"]], poly_df["fuel_efficiency_kmpl"], test_size=0.3, random_state=SEED)

def poly_model(degree):
    # Order matters: create powers -> scale them (keeps high powers numerically sane) -> fit OLS
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())

rows = []
for d in [1, 2, 3, 15]:
    model = ...          # TODO: poly_model(d) fitted on the training data
    train_mse = ...      # TODO
    test_mse = ...       # TODO
    rows.append({"degree": d, "train MSE": train_mse, "test MSE": test_mse})
pd.DataFrame(rows).round(3)

In [ ]:
# Overfitting is easiest to see when training data are scarce: fit on only 25 trips
small = poly_df.sample(25, random_state=3); rest = poly_df.drop(small.index)
grid = pd.DataFrame({"speed_kmph": np.linspace(20, 108, 400)})
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, d in zip(axes, [1, 2, 3, 15]):
    m = poly_model(d).fit(small[["speed_kmph"]], small["fuel_efficiency_kmpl"])
    ax.scatter(rest["speed_kmph"], rest["fuel_efficiency_kmpl"], s=8, color="lightgray", label="unseen")
    ax.scatter(small["speed_kmph"], small["fuel_efficiency_kmpl"], s=18, color="k", label="training")
    ax.plot(grid["speed_kmph"], m.predict(grid), color="C1", lw=2)
    unseen = mean_squared_error(rest["fuel_efficiency_kmpl"], m.predict(rest[["speed_kmph"]]))
    ax.set_title(f"degree {d}\nunseen MSE = {unseen:,.2f}"); ax.set_ylim(4, 22); ax.set_xlabel("speed (km/h)")
axes[0].set_ylabel("km/l"); plt.tight_layout(); plt.show()

**Questions**
1. Which degree underfits? Which overfits? What evidence in the table/plots supports each answer?
2. Why do we put `StandardScaler` *after* `PolynomialFeatures` here?
3. A friend says "degree 15 has the lowest training error, so it's the best model." Respond.
*Write your answers here.*

---
## A9 · Bias–variance: training vs validation behaviour
**Objective:** pick a degree with **5-fold cross-validation on the training set only**, and observe variance directly by refitting on many small samples.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
records = []
for d in range(1, 16):
    train_mse = ...   # TODO: fit poly_model(d) on Xp_train and compute training MSE
    cv_mse = ...      # TODO: -cross_val_score(poly_model(d), Xp_train, yp_train, cv=cv, scoring="neg_mean_squared_error").mean()
    records.append({"degree": d, "train MSE": train_mse, "CV MSE": cv_mse})
curve = pd.DataFrame(records)
best_degree = int(curve.loc[curve["CV MSE"].idxmin(), "degree"])
print("degree with lowest CV MSE:", best_degree)
plt.plot(curve["degree"], curve["train MSE"], "o-", label="training")
plt.plot(curve["degree"], curve["CV MSE"], "o-", label="5-fold CV")
plt.yscale("log"); plt.xlabel("degree"); plt.ylabel("MSE"); plt.legend(); plt.show()

final_poly = poly_model(best_degree).fit(Xp_train, yp_train)
print("test MSE of the chosen degree (used once):", round(mean_squared_error(yp_test, final_poly.predict(Xp_test)), 3))

In [ ]:
# Variance made visible: refit each model on 30 different random samples of 25 trips,
# and record its prediction at speed = 100 km/h (near the edge of the data).
query = pd.DataFrame({"speed_kmph": [100.0]})
spread = {}
for d in [1, 2, 9]:
    preds = []
    for rep in range(30):
        s = poly_df.sample(25, random_state=100 + rep)
        preds.append(poly_model(d).fit(s[["speed_kmph"]], s["fuel_efficiency_kmpl"]).predict(query)[0])
    spread[f"degree {d}"] = {"mean prediction": np.mean(preds), "std of predictions": np.std(preds)}
pd.DataFrame(spread).T.round(3)

**Questions**
1. Describe the shape of the two curves. Where is the model underfitting, and where is it overfitting?
2. In the resampling table, which degree has the largest spread of predictions? Which term of the bias–variance decomposition does that spread illustrate?
3. Why is it wrong to pick the degree by looking at the test MSE of every degree?

**Reflection:** state the bias–variance trade-off in one sentence of your own.
*Write your answers here.*

---
## A10 · Ridge and Lasso (introduction)
**Objective:** compare coefficients of Linear, Ridge and Lasso regression on the house data, and see regularisation tame an overfitting polynomial.

**Conventions in scikit-learn** (different from the classroom "SSE + penalty" form, so α values are *not* interchangeable):
- `Ridge`: $\|y-X\beta\|^2_2+\alpha\|\beta\|^2_2$  (intercept not penalised)
- `Lasso`: $\frac{1}{2n}\|y-X\beta\|^2_2+\alpha\|\beta\|_1$  (intercept not penalised)

Penalties depend on coefficient size, which depends on feature units → **always scale inside a Pipeline**.

In [ ]:
models = {
    "Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge (alpha=10)": make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "Lasso (alpha=1.5)": make_pipeline(StandardScaler(), Lasso(alpha=1.5)),
}
coefs, scores = {}, {}
for name, pipe in models.items():
    ...   # TODO: fit pipe on X_train, y_train; store pipe[-1].coef_ in coefs[name]
    ...   # TODO: store regression_report(y_test, pipe.predict(X_test)) in scores[name]
display(pd.DataFrame(coefs, index=FEATURES).round(3))
display(pd.DataFrame(scores).round(3))

In [ ]:
# Choose Ridge's alpha by cross-validation on the training set (never on the test set)
for alpha in [0.1, 1, 10, 100, 1000]:
    rmse = -cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=alpha)), X_train, y_train,
                            cv=cv, scoring="neg_root_mean_squared_error").mean()
    print(f"Ridge alpha={alpha:<6} CV RMSE = {rmse:.3f}")

In [ ]:
# Regularisation rescuing the degree-15 polynomial fitted on 25 trips
for name, last in [("OLS", LinearRegression()), ("Ridge alpha=1", Ridge(alpha=1.0))]:
    m = make_pipeline(PolynomialFeatures(15, include_bias=False), StandardScaler(), last)
    m.fit(small[["speed_kmph"]], small["fuel_efficiency_kmpl"])
    print(f"degree 15 + {name:<14} unseen MSE = {mean_squared_error(rest['fuel_efficiency_kmpl'], m.predict(rest[['speed_kmph']])):,.3f}")

**Questions**
1. Which coefficient did Lasso set exactly to zero? Why is that feature a natural candidate?
2. Compare the Ridge and Linear coefficients for `area_sqft` and `bedrooms`. What did Ridge do with these two correlated features?
3. Why would fitting Ridge **without** scaling be a problem in this dataset?
4. Explain, using the penalty term, why the degree-15 Ridge model behaves so much better on unseen data.
5. Which Ridge α does cross-validation prefer for the house data, and what does that tell you?
*Write your answers here.*

---
## A11 · Final Challenge — delivery time prediction
**Objective:** build, evaluate and improve a regression model **independently**. (Full brief and marking rubric: `08_Final_Regression_Challenge.docx`.)

**Dataset:** `delivery_time_challenge.csv` — 260 synthetic food deliveries. Target: `delivery_time_min`.

**Your steps**
1. Load and inspect (shape, dtypes, missing values, summary).
2. Identify features and target; split with `test_size=0.2, random_state=42`.
3. Train a `LinearRegression` baseline; report MAE, MSE, RMSE, R² on the test set.
4. Interpret two coefficients in plain language with units.
5. Propose **one** alternative (a transformation, polynomial terms, or regularisation). Compare with the baseline using 5-fold CV RMSE on the **training set**.
6. Evaluate only your chosen final model on the test set.
7. Comment on any sign of overfitting (compare training vs CV error).
8. Manual question: using your baseline coefficients, compute the prediction for one test row by hand.

In [ ]:
# TODO: your solution — add as many cells as you need

In [ ]:
# TODO: alternative model, CV comparison, final test evaluation, overfitting comment

**Write your interpretation here** (coefficients, model choice, overfitting evidence, manual calculation).

---
### End of notebook — summary
| idea | where you used it |
|---|---|
| OLS line & residuals | A2–A4 |
| Gradient descent | A5 |
| MAE / MSE / RMSE / R² | A6 |
| Multiple regression & interpretation | A7 |
| Polynomial features | A8 |
| Bias–variance, cross-validation | A9 |
| Ridge / Lasso | A10 |